## Importation des librairies

In [2]:
# Accès aux fichiers et dossiers

from pathlib import Path

# Analyse de données

import pandas as pd
import plotly as plt
import plotly.express as px
import plotly.figure_factory as ff
import folium

# Préparation de données pour le machine learning

from sklearn.preprocessing import StandardScaler

# Modélisation et évaluation

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


## EDA

In [3]:
# Chargement des données depuis le fichier CSV

# Données Uber pickups entre avril et septembre 2014

# Chemin vers le dossier contenant les fichiers

data_folder = Path("C:/Users/vavro/OneDrive/Desktop/Projet Uber Pickups/Data_2014")

# Récupérer tous les fichiers CSV

files = sorted(data_folder.glob("*.csv"))

print("Nombre de fichiers trouvés :", len(files))

# Charger chaque fichier

dfs = [pd.read_csv(file) for file in files]

# Fusionner les fichiers verticalement

df = pd.concat(dfs, ignore_index=True)

print("Dimensions du dataset fusionné :", df.shape)

df.head()

Nombre de fichiers trouvés : 6
Dimensions du dataset fusionné : (4534327, 4)


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [4]:
df.info()
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 4534327 entries, 0 to 4534326
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Date/Time  str    
 1   Lat        float64
 2   Lon        float64
 3   Base       str    
dtypes: float64(2), str(2)
memory usage: 138.4 MB


,Date/Time,Lat,Lon,Base
count,4534327,4.534327e+06,4.534327e+06,4534327
unique,260093,NaN,NaN,5
top,4/7/2014 20:21:00,NaN,NaN,B02617
freq,97,NaN,NaN,1458853
mean,NaN,4.073926e+01,-7.397302e+01,NaN
std,NaN,3.994991e-02,5.726670e-02,NaN
min,NaN,3.965690e+01,-7.492900e+01,NaN
25%,NaN,4.072110e+01,-7.399650e+01,NaN
50%,NaN,4.074220e+01,-7.398340e+01,NaN
75%,NaN,4.076100e+01,-7.396530e+01,NaN


In [5]:
# Conversion de la colonne 'Date/Time' en type datetime

date = pd.to_datetime(df['Date/Time'], format='%m/%d/%Y %H:%M:%S', errors="coerce")

In [6]:
# Vérification des valeurs manquantes dans le DataFrame

df.isnull().sum()

Date/Time    0
Lat          0
Lon          0
Base         0
dtype: int64

In [7]:
# Vérification des doublons dans le DataFrame

print(f"Nombre de doublons : {df.duplicated().sum()}")
print(f"Le dataset contient {df.duplicated().mean()*100:.2f}% doublons.")

Nombre de doublons : 82581
Le dataset contient 1.82% doublons.


In [8]:
# Calcul du nombre d'occurrences de chaque ligne

duplicate_counts = (
    df.groupby(df.columns.tolist(), dropna=False)
      .size()
      .reset_index(name="count")
)

# Affichage des lignes apparaissant plusieurs fois

duplicate_counts[duplicate_counts["count"] > 1].head()

print(duplicate_counts[duplicate_counts['count'] > 1].head(20))
print('--------------------------------------------------------')
print(f"Nombre de lignes avec 2 occurrences : {duplicate_counts[duplicate_counts['count'] == 2].shape[0]}")
print(f"Nombre de lignes avec 3 occurrences : {duplicate_counts[duplicate_counts['count'] == 3].shape[0]}")
print(f"Nombre de lignes avec 4 occurrences : {duplicate_counts[duplicate_counts['count'] == 4].shape[0]}")


              Date/Time      Lat      Lon    Base  count
61     4/1/2014 0:27:00  40.7362 -73.9768  B02617      2
80     4/1/2014 0:35:00  40.7078 -74.0120  B02682      2
83     4/1/2014 0:37:00  40.7613 -73.9791  B02617      2
96     4/1/2014 0:43:00  40.7286 -74.0001  B02617      2
108    4/1/2014 0:48:00  40.7136 -73.9990  B02617      2
181   4/1/2014 10:06:00  40.6943 -73.9454  B02598      2
327   4/1/2014 10:26:00  40.7480 -73.9929  B02682      2
375   4/1/2014 10:32:00  40.6949 -74.1781  B02512      2
485   4/1/2014 10:43:00  40.7622 -73.9741  B02682      2
621   4/1/2014 11:01:00  40.7883 -74.0456  B02512      2
665   4/1/2014 11:06:00  40.8275 -74.0468  B02512      2
884   4/1/2014 11:30:00  40.6991 -73.9556  B02682      2
963   4/1/2014 11:38:00  40.7448 -73.9828  B02617      2
1030  4/1/2014 11:45:00  40.7291 -73.9955  B02682      2
1170  4/1/2014 12:02:00  40.7852 -74.0220  B02512      2
1219  4/1/2014 12:10:00  40.8289 -73.9451  B02512      2
1321  4/1/2014 12:25:00  40.668

Sur les 4 534 327 observations, 82 581 doublons ont été identifiés, soit environ 1,82 % du dataset. Je choisis de les conserver, car dans une ville aussi dense que New York, plusieurs demandes de course Uber peuvent avoir lieu au même endroit et au même moment. Ces doublons apparents ne correspondent donc pas nécessairement à des erreurs. Leur suppression pourrait entraîner une perte d'informations pertinentes et sous-estimer la demande réelle de courses.

# KMeans

In [9]:
# Standardisation des valeurs numériques

# Définition des variables X

X = df[['Lat', 'Lon']]

# Initialisation de Standardscaler

sc = StandardScaler()

# Application de Standardscaler sur les variables X

X = sc.fit_transform(X)

# Visualisation des variables standardisées

X[:5]


array([[ 0.74441341,  0.31640287],
       [-0.3144126 , -1.07358489],
       [-0.191759  , -0.24937104],
       [ 0.48909366, -0.07998811],
       [ 0.50411247,  0.01430754]])

In [10]:
# Initialisation du modèle KMeans

kmeans = KMeans (n_clusters=3, random_state=0)

# Application du modèle sur le dataset

kmeans.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary <random_state>`.",0
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'
Name,Type,Value
"cluster_centers_ cluster_centers_: ndarray of shape (n_clusters, n_features)Coordinates of cluster centers. If the algorithm stops before fullyconverging (see ``tol`` and ``max_iter``), these will not beconsistent with ``labels_``.","ndarray[float64](3, 2)","[[

In [11]:
# Identification des coordonnées des centroides

cluster_centers = kmeans.cluster_centers_

print("Coordonnées des centroides standardisées :")
print(cluster_centers)

# Revenir aux coordonnées originales

cluster_centers_reel = sc.inverse_transform(cluster_centers)

print('Coordonnées réelles des centroides :')
print(cluster_centers_reel)

Coordonnées des centroides standardisées :
[[ 0.73201784  0.13393127]
 [-0.52243946 -0.36312924]
 [-1.79687032  3.49612619]]
Coordonnées réelles des centroides :
[[ 40.7685048  -73.96534954]
 [ 40.71838935 -73.99381456]
 [ 40.66747595 -73.77280777]]


In [12]:
# Création d'un heatmap

labels = ["Lat", "Lon"]

matrix = pd.DataFrame(cluster_centers_reel, columns = labels)

fig = px.imshow(
    matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Heatmap des centroïdes K-means (coordonnées réelles)"
)

fig.update_layout(
    xaxis_title="Variables",
    yaxis_title="Clusters"
)

fig.show()


Les clusters 0 et 2 ont des latitudes et des longitudes assez proches. Cela signifie que leurs centres géographiques sont proches.

## KMeans 

### Méthode Elbow

In [13]:
# Calcul de l'inertie (WCSS) pour différentes valeurs de K, de 1 à 10,
# afin de déterminer le nombre optimal de clusters avec la méthode du coude

wcss = []
k = []

for i in range (1, 11) :
    kmeans= KMeans(n_clusters=i, random_state = 0, n_init='auto')
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
    k.append(i)
    print("WCSS pour K={} ---->{}".format(i, wcss[-1]))

WCSS pour K=1 ---->9068654.000000432
WCSS pour K=2 ---->6863611.161457552
WCSS pour K=3 ---->4486544.12411573
WCSS pour K=4 ---->3676061.8867160464
WCSS pour K=5 ---->3264803.7057841388
WCSS pour K=6 ---->2958956.8720650785
WCSS pour K=7 ---->2410987.125891319
WCSS pour K=8 ---->1962500.1067161541
WCSS pour K=9 ---->1739326.1371818678
WCSS pour K=10 ---->1587707.2890594292


In [14]:
# Visualisation de l'inertie en fonction du nombre de clusters
# afin d'identifier le nombre optimal de clusters avec la méthode du coude

wcss_frame = pd.Series(wcss)
k_frame = pd.Series(k)

fig=px.line(
    x=k_frame,
    y=wcss_frame
)

fig.update_layout(
    yaxis_title="Inertia",
    xaxis_title="Clusters",
    title="Inertia per cluster"
)

fig.show()

La méthode du coude montre une forte diminution de l’inertie entre K = 2 et K = 4, puis une baisse plus progressive.

## KMeans

### Méthode Silhouette

In [15]:
# Évaluation de la qualité du clustering avec la méthode silhouette
# pour différentes valeurs de K, de 2 à 10

sil = []
k = []

for i in range(2,11):
    kmeans = KMeans(n_clusters=i, random_state=0, n_init='auto')
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_,  sample_size=5000, random_state=0)
    sil.append(score)
    k.append(i)
    print(f"Silhouette score pour K={i} est : {score:.4f}")

Silhouette score pour K=2 est : 0.3633
Silhouette score pour K=3 est : 0.4083
Silhouette score pour K=4 est : 0.4257
Silhouette score pour K=5 est : 0.3801
Silhouette score pour K=6 est : 0.3757
Silhouette score pour K=7 est : 0.4517
Silhouette score pour K=8 est : 0.4616
Silhouette score pour K=9 est : 0.4336
Silhouette score pour K=10 est : 0.4459


In [16]:
# Visualisation des scores silhouette pour chaque valeur de K
# afin de comparer la qualité des clusters et d'identifier le nombre optimal de clusters

cluster_scores = pd.Series(sil)
k_frame = pd.Series(k)

fig=px.bar(x=k, y=cluster_scores)

fig.update_layout(
    yaxis_title="Silhouette Score",
    xaxis_title="# Clusters",
    title="Silhouette Score per cluster"
)

fig.show()

In [17]:
# Initialisation du modèle KMeans avec K = 8

kmeans = KMeans (n_clusters=8, random_state=0)

# Application du modèle sur le dataset

df["cluster"] = kmeans.fit_predict(X)

In [18]:
# Identification des coordonnées des centroides

cluster_centers = kmeans.cluster_centers_

cluster_centers_reel_8 = sc.inverse_transform(cluster_centers)

print(cluster_centers_reel_8)

[[ 40.76626845 -73.97337568]
 [ 40.73087395 -73.99731515]
 [ 40.65902744 -73.76222711]
 [ 40.69733468 -74.20375979]
 [ 40.62882387 -73.97708046]
 [ 40.76343272 -73.86740276]
 [ 40.69553841 -73.96352798]
 [ 40.88633038 -73.8913507 ]]


## Analyse des hot zones

In [19]:
# Création de la carte centrée sur les coordonnées moyennes des points de prise en charge

m = folium.Map(
    location=[df["Lat"].mean(),
              df["Lon"].mean()],
    zoom_start=11
)

colors = ["red", "blue", "green", "purple", "orange", "darkred", "black", "rose"]

# Échantillonnage aléatoire de 5 000 observations maximum
# afin de réduire le temps de calcul et d'accélérer l'affichage de la carte

df_sample = df.sample(
    n=min(5000, len(df)),
    random_state=42
)

for _, row in df_sample.iterrows():
    # Ajout d'un marqueur circulaire pour chaque point de prise en charge
    folium.CircleMarker(
        # Position géographique du point à partir de la latitude et de la longitude
        location=[row["Lat"], row["Lon"]],
        # Définition de la taille du marqueur
        radius=2,
        # Attribution d'une couleur selon le cluster de l'observation
        color=colors[int(row["cluster"])],
        # Remplissage du cercle
        fill=True,
        # Définition de la transparence du marqueur
        fill_opacity=0.6,
        # Affichage du numéro du cluster au clic sur le marqueur
        popup=f"Cluster {row['cluster']}"
    ).add_to(m)
    
    # Affichage de la carte interactive
m

In [20]:
# Conversion de la colonne Date/Time en datetime

df["Date/Time"] = pd.to_datetime(df["Date/Time"])

# Extraction du jour de la semaine

df['Day_of_week'] = (df["Date/Time"].dt.day_name()
)

# Extraction de l'heure

df['Hour'] = df["Date/Time"].dt.hour

df.head()

,Date/Time,Lat,Lon,Base,cluster,Day_of_week,Hour
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,0,Tuesday,0
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,1,Tuesday,0
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,1,Tuesday,0
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,0,Tuesday,0
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,0,Tuesday,0


In [21]:
# Calcul du nombre de prises en charge par cluster

demandes_hot_zones = (df.groupby("cluster").size().reset_index(name="Number of pickups"))
                     
demandes_hot_zones = demandes_hot_zones.sort_values(by="Number of pickups", ascending=False)

demandes_hot_zones["Percentage"] = (
    demandes_hot_zones["Number of pickups"]
    / demandes_hot_zones["Number of pickups"].sum()
    * 100
).round(2)
             
display(demandes_hot_zones.set_index("cluster"))
         

,Number of pickups,Percentage
cluster,,
1,1866647,41.17
0,1672352,36.88
6,510461,11.26
5,171958,3.79
2,134557,2.97
4,71639,1.58
7,62390,1.38
3,44323,0.98


In [22]:
# Analyse des jours les plus demandés par cluster 
# Création d'un tableau croisé indiquant le nombre de prises en charge pour chaque combinaison de cluster et de jour de la semaine

demand_by_cluster = pd.crosstab(
    df["cluster"],
    df["Day_of_week"]
)

# Définition de l'ordre chronologique des jours de la semaine

days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Réorganisation des colonnes selon l'ordre des jours
# et ajout d'une valeur 0 si un jour est absent des données

demand_by_cluster = demand_by_cluster.reindex(columns=days_order, fill_value=0)

print(demand_by_cluster)

# Identification du jour qui compte le plus de prises en charge
# pour chaque cluster

most_demanded_day = demand_by_cluster.idxmax(axis=1)

# Création d'un tableau récapitulatif

summary = pd.DataFrame({
    "Most demanded day": most_demanded_day,
    "Number of pickups": demand_by_cluster.max(axis=1)
})

print(summary)

Day_of_week  Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
cluster                                                                    
0            210339   272242     283984    297550  270248    197230  140759
1            205045   267304     286402    318172  319723    278512  191489
2             22638    17139      17425     18801   18788     15464   24302
3              7271     6157       6067      7056    6417      4553    6802
4              8277     8701       8918      9198   10620     14167   11758
5             25515    22772      22838     27970   24411     20948   27504
6             54869    60950      62393     67606   81418    104276   78949
7              7518     8524       8461      8792    9514     10964    8617
        Most demanded day  Number of pickups
cluster                                     
0                Thursday             297550
1                  Friday             319723
2                  Sunday              24302
3              

In [23]:
# Visualisation du nombre de prises en charge par jour et par cluster
# à l'aide d'une heatmap

fig = px.imshow(
    demand_by_cluster,
    labels={
        "x": "Day of week",
        "y": "Cluster",
        "color": "Number of pickups"
    },
    x=demand_by_cluster.columns,
    y=demand_by_cluster.index,
    color_continuous_scale="YlOrRd",
    text_auto=True,
    aspect="auto",
    title="Weekly Pickup Demand by Cluster"
)

fig.show()

Les clusters 0 et 1 concentrent les volumes de prises en charge les plus importants, avec une activité particulièrement élevée en milieu et en fin de semaine. Le cluster 1 atteint son maximum le vendredi, avec environ 320 000 prises en charge, tandis que le cluster 0 culmine le jeudi, avec environ 298 000 prises en charge. Le cluster 6 se distingue par une forte hausse le samedi, avec environ 104 000 prises en charge. À l'inverse, les clusters 3 et 7 enregistrent des volumes nettement plus faibles.

In [24]:
# Analyse des heures de pointe pour chaque cluster

# Création d'un tableau croisé indiquant le nombre de prises en charge
# pour chaque combinaison de cluster et d'heure

demand_by_hour = pd.crosstab(
    df["cluster"],
    df["Hour"]
)

# Réorganisation des colonnes pour afficher toutes les heures,
# de 0 h à 23 h, dans l'ordre chronologique
# Ajout d'une valeur 0 si une heure est absente des données

demand_by_hour = demand_by_hour.reindex(columns=range(24), fill_value=0)

print(demand_by_hour)

# Identification de l'heure qui enregistre le plus de prises en charge
# pour chaque cluster

most_demanded_hour = demand_by_hour.idxmax(axis=1)

# Création d'un tableau récapitulatif

summary_hours = pd.DataFrame({
    "Most demanded hour": most_demanded_hour,
    "Number of pickups": demand_by_hour.max(axis=1)
})

print(summary_hours)

Hour        0      1      2      3      4      5      6      7      8      9   \
cluster                                                                         
0        23236  14263  10279  13855  16503  26425  54036  78774  75246  62127   
1        53394  36546  24086  21776  20571  30157  52722  74102  76121  59115   
2         3114   1116    591    716   3247   5260   5899   3829   2370   2776   
3         1232    585    206    287   1019   1116   1235   1355   1240   1225   
4         1932   1232    895   1113   1424   2120   2941   3468   3348   3042   
5         2272   1330   1076   1499   1778   2655   4639   6095   7645   8627   
6        17164  11258   7992   8090   9620  14563  19640  23194  22272  20515   
7         1492    897    740    951   1068   1643   2101   2277   2262   2540   

Hour     ...     14      15      16      17      18      19      20      21  \
cluster  ...                                                                  
0        ...  88726  113939  13

In [25]:
# Visualisation des heures de pointe par cluster à l'aide d'une heatmap

fig = px.imshow(
    demand_by_hour,
    labels={
        "x": "Hour of day",
        "y": "Cluster",
        "color": "Number of pickups"
    },
    x=demand_by_hour.columns,
    y=demand_by_hour.index,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="Hourly Pickup Demand by Cluster"
)

fig.update_layout(
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)

fig.show()

Les clusters 0 et 1 concentrent les volumes de prises en charge les plus élevés, avec une forte hausse à partir de 14 h et des pics en fin d'après-midi, entre 16 h et 18 h. Le cluster 0 reste particulièrement actif jusqu'en soirée. Le cluster 6 présente une demande plus modérée, qui augmente progressivement au cours de la journée, tandis que les autres clusters enregistrent des volumes nettement plus faibles..

In [26]:
# Analyse du nombre de prises en charge par cluster, jour et heure

demand = (
    df.groupby(["cluster", "Day_of_week", "Hour"])
    .size()
    .rename("Number of pickups")
    .reset_index()
)

demand.head()

,cluster,Day_of_week,Hour,Number of pickups
0,0,Friday,0,3476
1,0,Friday,1,1994
2,0,Friday,2,1416
3,0,Friday,3,2262
4,0,Friday,4,2827


In [27]:
# Identification de l'heure de pointe pour chaque cluster et chaque jour

# Récupération de l'index de la ligne contenant le nombre maximal
# de prises en charge pour chaque combinaison cluster / jour

idx = demand.groupby(["cluster", "Day_of_week"])["Number of pickups"].idxmax()

peak_hours = demand.loc[idx].copy()

# Définition de l'ordre chronologique des jours de la semaine

days_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

# Conversion de la colonne des jours en catégorie ordonnée
# pour garantir le bon ordre d'affichage

peak_hours["Day_of_week"] = pd.Categorical(
    peak_hours["Day_of_week"],
    categories=days_order,
    ordered=True
)

peak_hours = peak_hours.sort_values(
    ["cluster", "Day_of_week"]
)

peak_hours

,cluster,Day_of_week,Hour,Number of pickups
41,0,Monday,17,19000
137,0,Tuesday,17,25450
161,0,Wednesday,17,25101
113,0,Thursday,17,24808
18,0,Friday,18,21896
66,0,Saturday,18,16680
88,0,Sunday,16,11359
209,1,Monday,17,16263
305,1,Tuesday,17,23173
329,1,Wednesday,17,23535


In [28]:
# Visualisation des heures de pointe par cluster et par jour

peak_matrix = peak_hours.pivot(
    index="Day_of_week",
    columns="cluster",
    values="Hour"
).reindex(days_order)

fig = px.imshow(
    peak_matrix,
    labels={
        "x": "Cluster",
        "y": "Day of week",
        "color": "Peak hour"
    },
    x=peak_matrix.columns,
    y=peak_matrix.index,
    color_continuous_scale="Turbo",
    text_auto=True,
    aspect="auto",
    title="Peak Pickup Hour by Cluster and Day"
)

fig.show()

Les clusters 0 et 1 présentent des pics principalement en fin d'après-midi, entre 17 h et 18 h, avec des pointes plus tardives le vendredi et le samedi. Les clusters 5, 6 et 7 connaissent également une activité importante en soirée, souvent entre 20 h et 22 h. À l'inverse, les clusters 2 et 3 affichent des heures de pointe plus stables au cours de la semaine, autour de 14 h et 15 h.
Ces résultats suggèrent que la demande Uber dépend à la fois de la localisation et du jour de la semaine.

# Conclusion

L’analyse met en évidence huit zones de demandes Uber. Les principales zones avec une forte demande se situe à Manhattant et Brooklyn (zones 0, 1, 6). Les heures de pointe varient selon les zones et les jours de la semaine, avec une concentration générale des demandes en fin de journée. Voici les recommandations pour chaque zone :

Zone 0 : Manhattan, principalement Midtown et l'axe vers Upper Manhattan : le jour le plus demandé est le jeudi -> Renforcer la disponibilité tous les jours en fin d'après-midi entre 16h et 18h.

Zone 1 : Manhattan, l'axe vers West village et East village, lower Manhattan : le jour le plus demandé est le vendredi -> Renforcer la disponibilité vers 17h end début de semaine et à partir de 21h en fin de semaine, notament les vendredis et samedis.

Zone 2 : Zones périphériques dispersées, notamment le sud de Queens et Long Island : le jour le plus demandé est le dimanche -> Renforcer la disponibilité vers 14h du lundi au samedi, 19h le dimanche.

Zone 3 : Secteurs à l'ouest de Manhattan et zones du New Jersey : le jour le plus demandé est le lundi -> Prévoir une couverture régulière l'après-midi, l'heure de pointe est à 15h.

Zone 4 : Sud de Brooklyn : le jour le plus demandé est le samedi -> Prévoir une couverture régulière à 7h en semaine et vers 20h à 21h du vendredi au dimanche.

Zone 5 : Nord de Queens : le jour le plus demandé est le jeudi -> Renforcer la disponibilité en soirée tout au long de la semaine, à 17h le samedi.

Zone 6 : Nord de Brooklyn : le jour le plus demandé est le samedi -> Prévoir une couverture tardive du mardi au dimanche, à partir de 21h.

Zone 7 : Manhattan (Washington Heights) : le jour le plus demandé est le samedi -> Renforcer la disponibilité entre 20h et 21h du mardi au samedi et entre 14h et 15h le lundi et le dimanche.

Ces résultats montrent que la demande Uber varie selon la localisation, le jour de la semaine et l'heure. Ils pourraient contribuer à optimiser la répartition des chauffeurs en fonction des périodes de forte activité et des besoins spécifiques à chaque zone. Toutefois, pour aller plus loin et évaluer concrètement l'amélioration du service, il serait nécessaire de disposer de données détaillées sur le temps d'attente entre la demande d'une course et la prise en charge effective du client. Ces informations permettraient d'évaluer si une meilleure répartition des chauffeurs contribue à réduire les délais d'attente et à se rapprocher de l'objectif de moins de 7 minutes.